# Ground truth rendering and comparison

Draws every `data/plan*/truth.json` (metres: hall, lobby, wall, doors, booths, seats, stage, kids, pilasters, fixtures) next to its `layout.png`, and as an overlay on it. `data/plan*/frame.json` gives the pixel scale and origin of each plan. The comparison pages are saved to `output/truth` and to one multi-page PDF.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
from IPython.display import Image, display
from PIL import Image as PILImage

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output/truth"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BGR colours
HALL = (214, 244, 255)
LOBBY = (252, 238, 222)
BOOTH = {"goods": (150, 215, 150), "fnb": (90, 180, 250), "fit": (230, 170, 200)}
TABLE = (230, 160, 110)
KIDS = (250, 215, 225)
STAGE = (80, 80, 170)
FIXTURE = (190, 190, 190)
CLOSED = (200, 200, 250)
DOOR = (200, 60, 230)


In [ ]:
def load_plan(plan_name):
    truth = json.loads((DATA_DIR / plan_name / "truth.json").read_text())
    frame = json.loads((DATA_DIR / plan_name / "frame.json").read_text())
    return truth, frame


class Px:
    """Metres to layout.png pixels: pixel = (metres - 1) * px_per_m + origin."""

    def __init__(self, frame):
        self.s = frame["px_per_m"]
        self.ox, self.oy = frame["origin_px"]

    def pt(self, x, y):
        return int(round((x - 1) * self.s + self.ox)), int(round((y - 1) * self.s + self.oy))

    def rect(self, r):
        if "cx" in r:
            return self.pt(r["cx"] - r["w"] / 2, r["cy"] - r["h"] / 2), self.pt(r["cx"] + r["w"] / 2, r["cy"] + r["h"] / 2)
        return self.pt(r["x0"], r["y0"]), self.pt(r["x1"], r["y1"])


def draw_label(img, text, p0, p1, color=(0, 0, 0)):
    w, h = p1[0] - p0[0], p1[1] - p0[1]
    scale = min(0.65, max(0.2, min(h, w * 0.6) / 38.0))
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)
    if tw > max(2, w - 2):
        scale = max(0.18, scale * (w - 2) / tw)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)
    org = (p0[0] + max(1, (w - tw) // 2), p0[1] + (h + th) // 2)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)


def draw_truth(canvas, truth, frame, filled=True):
    """Draw the metre truth in layout pixels. filled=False draws outlines only, for the overlay."""
    px = Px(frame)

    def box(r, color, thickness=1):
        p0, p1 = px.rect(r)
        if filled:
            cv2.rectangle(canvas, p0, p1, color, -1)
            cv2.rectangle(canvas, p0, p1, tuple(int(c * 0.5) for c in color), 1)
        else:
            cv2.rectangle(canvas, p0, p1, color, max(1, thickness))
        return p0, p1

    if filled:
        cv2.fillPoly(canvas, [np.array([px.pt(*p) for p in truth["hall"]], np.int32)], HALL)
        cv2.fillPoly(canvas, [np.array([px.pt(*p) for p in truth["lobby"]], np.int32)], LOBBY)
    cv2.polylines(canvas, [np.array([px.pt(*p) for p in truth["hall"]], np.int32)], True, (0, 0, 0), 2)
    cv2.polylines(canvas, [np.array([px.pt(*p) for p in truth["lobby"]], np.int32)], True, (60, 60, 60), 2)
    for c in truth["closed"]:
        box(c, CLOSED, 2)
    kids = truth["kids"]
    box(kids, KIDS, 2)
    stage = truth["stage"]
    if any(stage[k] for k in ("x0", "x1", "y0", "y1")):
        box(stage, STAGE, 3)
    for f in truth["fixtures"]:
        p0, p1 = box(f, FIXTURE, 2)
        draw_label(canvas, f["label"], p0, p1)
    for b in truth["booths"]:
        p0, p1 = box(b, BOOTH.get(b["type"], (200, 200, 200)), 2)
        draw_label(canvas, b["id"], p0, p1)
    for t in truth["tables"]:
        box(t, TABLE, 1)
    for p in truth["pilasters"]:
        box(p, (0, 0, 0), 3) if p["kind"] != "block" else box(p, (60, 60, 60), 3)
    for c in truth["columns"]:
        cv2.circle(canvas, px.pt(c["x"], c["y"]), max(2, int(c["d"] / 2 * px.s)), (0, 0, 0), -1 if filled else 2)
    box(truth["wall"], (0, 0, 0), 3)
    for d in truth["doors"]:
        p0, p1 = box(d, DOOR, 3)
        draw_label(canvas, d["id"][-1] + d["role"][0].upper(), p0, p1)
    for o in truth["outerDoors"]:
        cv2.line(canvas, px.pt(o["x"], o["y0"]), px.pt(o["x"], o["y1"]), DOOR, 4)
    for g in truth["gates"]:
        cv2.circle(canvas, px.pt(g["sx"], g["sy"]), 9, (0, 160, 0), -1)
    e = truth["exitPt"]
    cv2.circle(canvas, px.pt(e["sx"], e["sy"]), 9, (0, 0, 220), -1)
    return canvas


def render_truth_canvas(frame, truth, w, h):
    canvas = np.full((h, w, 3), 255, dtype=np.uint8)
    step = max(2, int(round(frame["px_per_m"] * 5)))   # a grid line every 5 m
    for pos in range(0, max(w, h), step):
        cv2.line(canvas, (pos, 0), (pos, h), (228, 228, 228), 1)
        cv2.line(canvas, (0, pos), (w, pos), (228, 228, 228), 1)
    return draw_truth(canvas, truth, frame, filled=True)


def render_truth_overlay(orig, truth, frame):
    return draw_truth(orig.copy(), truth, frame, filled=False)


In [ ]:
def make_side_by_side(plan_name, orig, rend, summary, target_w=1200):
    H, W = orig.shape[:2]
    s = target_w / W
    target_h = int(round(H * s))
    r_orig = cv2.resize(orig, (target_w, target_h), interpolation=cv2.INTER_AREA)
    r_rend = cv2.resize(rend, (target_w, target_h), interpolation=cv2.INTER_AREA)
    bar_h = 45
    combo = np.full((target_h + bar_h, target_w * 2 + 20, 3), 245, dtype=np.uint8)
    combo[bar_h:bar_h + target_h, 0:target_w] = r_orig
    combo[bar_h:bar_h + target_h, target_w + 20:target_w * 2 + 20] = r_rend
    cv2.putText(combo, f"Original Image: {plan_name}/layout.png ({W}x{H})", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (40, 40, 40), 2, cv2.LINE_AA)
    cv2.putText(combo, f"Truth Render: {plan_name}/truth.json ({summary})", (target_w + 40, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 110, 200), 2, cv2.LINE_AA)
    return combo


def make_overlay_page(plan_name, overlay, summary, target_w=1600):
    H, W = overlay.shape[:2]
    tw = min(target_w, W)
    th = int(round(H * tw / W))
    r_over = cv2.resize(overlay, (tw, th), interpolation=cv2.INTER_AREA)
    bar_h = 45
    page = np.full((th + bar_h, tw, 3), 245, dtype=np.uint8)
    page[bar_h:bar_h + th, 0:tw] = r_over
    cv2.putText(page, f"Truth Overlay on Original Image: {plan_name} ({summary}, {W}x{H})",
                (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 110, 200), 2, cv2.LINE_AA)
    return page


def summarize(truth):
    kinds = {}
    for b in truth["booths"]:
        kinds[b["type"]] = kinds.get(b["type"], 0) + 1
    booths = ", ".join(f"{n} {k}" for k, n in sorted(kinds.items()))
    walls = 1 + sum(1 for p in truth["pilasters"] if p["kind"] == "wall")
    return f"{booths}; {len(truth['tables'])} seats; {walls} walls; {len(truth['doors'])} doors"


def render_plan(plan_name, display_width=1000):
    img_path = DATA_DIR / f"{plan_name}/layout.png"
    if not (DATA_DIR / plan_name / "truth.json").exists() or not (DATA_DIR / plan_name / "frame.json").exists() or not img_path.exists():
        print(f"Skipping {plan_name}: missing truth.json, frame.json or layout.png")
        return None
    truth, frame = load_plan(plan_name)
    orig = cv2.imread(str(img_path))
    H, W = orig.shape[:2]
    summary = summarize(truth)
    print(f"=== {plan_name} Ground Truth ===")
    print(f"Resolution: {W} x {H} px | {frame['px_per_m']} px/m | world {truth['meta']['world']['w']} x {truth['meta']['world']['h']} m")
    print(summary)
    canvas = render_truth_canvas(frame, truth, W, H)
    overlay = render_truth_overlay(orig, truth, frame)
    sbs = make_side_by_side(plan_name, orig, canvas, summary)
    p_out = OUTPUT_DIR / plan_name
    p_out.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(p_out / "render.png"), canvas)
    cv2.imwrite(str(p_out / "overlay.png"), overlay)
    cv2.imwrite(str(p_out / "side_by_side.png"), sbs)
    if display_width:
        display(Image(data=cv2.imencode(".png", cv2.resize(sbs, (display_width, int(sbs.shape[0] * display_width / sbs.shape[1]))))[1].tobytes()))
    return {"plan": plan_name, "summary": summary}


In [ ]:
res1 = render_plan("plan1")

In [ ]:
res2 = render_plan("plan2")

In [ ]:
res3 = render_plan("plan3")

In [ ]:
res5 = render_plan("plan5")

In [ ]:
res6 = render_plan("plan6")

In [ ]:
# Export all ground truth comparisons into a single multi-page PDF
def export_all_pdf(pdf_path="output/truth_comparison.pdf"):
    out_pdf = ROOT / pdf_path
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    plans = [p.parent.name for p in sorted(DATA_DIR.glob("plan*/truth.json")) if (p.parent / "frame.json").exists()]
    pdf_pages = []
    for pname in plans:
        sbs_path = OUTPUT_DIR / pname / "side_by_side.png"
        render_plan(pname, display_width=None)
        if sbs_path.exists():
            pdf_pages.append(PILImage.open(sbs_path).convert("RGB"))
        truth, frame = load_plan(pname)
        orig = cv2.imread(str(DATA_DIR / pname / "layout.png"))
        over_page = make_overlay_page(pname, render_truth_overlay(orig, truth, frame), summarize(truth))
        pdf_pages.append(PILImage.fromarray(cv2.cvtColor(over_page, cv2.COLOR_BGR2RGB)))
    if pdf_pages:
        pdf_pages[0].save(out_pdf, save_all=True, append_images=pdf_pages[1:], resolution=150)
        print(f"Saved {out_pdf} ({len(pdf_pages)} pages, {out_pdf.stat().st_size / (1024 * 1024):.2f} MB)")
        mirror_pdf = OUTPUT_DIR / "truth_comparison.pdf"
        if mirror_pdf.resolve() != out_pdf.resolve():
            import shutil
            shutil.copy2(out_pdf, mirror_pdf)
            print(f"Mirrored to {mirror_pdf}")


export_all_pdf()
